# 04 · Dataset: scenes, splits, normalisation and training windows

Everything the training notebooks need to know about the data is decided here and saved to `data/tables/`:

- which scenes are used, and which are left out and why
- the split into train, validation and test, by ground, so the same area is never in two splits
- whether the sources are on the same scale, and the band normalisation
- how training windows are drawn: size, ground resolution, how often a window is centred on a burn
- the fixed windows that validation and test are scored on

No tiles are written to disk. Training windows are read from the GeoTIFFs while training (`burnscar/dataset.py`): a window at 10 m is a larger area of a 2.5 m scene averaged down, a window at 1 m a smaller area resampled up. So the model sees every scene at many resolutions, and the resolution range or the augmentations can change between runs without rebuilding anything.

Needs the v3 masks from notebooks 02 and 03. Run it one cell at a time; the decisions are in capital letters at the top of the cells, with the reason next to them.

## 1 · The scenes

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, glob, json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch, Rectangle
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import burnscar.labels as L
import burnscar.dataset as D
importlib.reload(L); importlib.reload(D)

TABLES = "data/tables"
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
C_BLUE, C_ORANGE, C_GREY, INK2 = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
SRC_NAME = {"phisat2_sim": "PhiSat-2 sim", "heo": "HEO", "s2_test": "Sentinel-2 test"}
SRC_COLOR = {"phisat2_sim": C_BLUE, "heo": C_ORANGE, "s2_test": C_GREY}
plt.rcParams.update({"axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "axes.edgecolor": "#8a8984", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2})
CMAP = ListedColormap([L.V3_COLORS[v] for v in L.V3_VALUES])
NORM = BoundaryNorm([-0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 200, 254.5, 255.5], CMAP.N)
SPLIT_COLOR = {"train": C_BLUE, "val": C_ORANGE, "test": "#b3261e", "excluded": "#c9c8c3"}
TRAIN_CMAP = ListedColormap([L.V3_COLORS[v] for v in (L.CLEAR, L.FRESH, L.OLD, L.CLOUD, L.SHADOW, L.WATER)])
TRAIN_CMAP.set_bad(L.V3_COLORS[L.IGNORE])                  # left out (ignore, no data) shows pink

In [ ]:
# [2] every scene that has an image and a v3 mask, with its footprint and class counts
rows, missing = [], []
for src, pattern in [("heo", "data/heo/images/*_img.tif"), ("phisat2_sim", "data/phisat2_sim/images/*_img.tif"),
                     ("s2_test", "data/s2_test/images/*_s2.tif")]:
    for img in sorted(glob.glob(pattern)):
        sid = os.path.basename(img).rsplit("_", 1)[0]
        msk = f"data/{src}/masks_v3/{sid}_mask.tif"
        if not os.path.exists(msk):
            missing.append(sid)
            continue
        with rasterio.open(img) as s:
            lon0, lat0, lon1, lat1 = transform_bounds(s.crs, "EPSG:4326", *s.bounds, densify_pts=21)
            rows.append(dict(scene=sid, source=src, img=img, mask=msk, px_m=round(abs(s.res[0]), 3),
                             height=s.height, width=s.width, epsg=s.crs.to_epsg(),
                             lon0=lon0, lat0=lat0, lon1=lon1, lat1=lat1,
                             acq=L.parse_acq_datetime(sid), scale=L.reflectance_scale(img)))
scenes = pd.DataFrame(rows)
scenes["level"] = np.where(scenes.source != "heo", "L1C",
                           scenes.scene.str[6:10].map({"1CST": "L1C", "2AST": "L2A"}).fillna("?"))
counts = pd.concat([pd.read_csv(f) for f in (f"{TABLES}/labels_v3_from_v2.csv", f"{TABLES}/labels_v3_heo_products.csv")
                    if os.path.exists(f)], ignore_index=True).drop_duplicates("scene", keep="last").set_index("scene")
for n in L.V3_NAMES.values():
    scenes[f"{n}_px"] = scenes.scene.map(counts[f"v3_{n}"]).fillna(0).astype(np.int64)

def stac_quality(sid):
    """Geolocation error (larger of the across- and along-track GCP RMS, metres), off-nadir angle and sun
    elevation from the product metadata. Only the products in data/heo/products have it."""
    files = sorted(glob.glob(f"data/heo/products/{sid}/**/*.json", recursive=True), key=len)
    if not files:
        return {}
    p = json.load(open(files[0])).get("properties", {})
    rms = [p[k] for k in ("iride:rms_gcp_displacement_act", "iride:rms_gcp_displacement_alt") if p.get(k) is not None]
    return {"geo_rms_m": max(rms) if rms else np.nan, "off_nadir": p.get("view:off_nadir", np.nan),
            "sun_elev": p.get("view:sun_elevation", np.nan)}

quality = pd.DataFrame([stac_quality(s) if src == "heo" else {} for s, src in zip(scenes.scene, scenes.source)],
                       index=scenes.index, columns=["geo_rms_m", "off_nadir", "sun_elev"])
scenes = pd.concat([scenes, quality], axis=1)

# country of every PhiSat-2 scene: the scene name starts with its EFFIS fire id, and the simulation run's index has the country
country = pd.read_csv("data/phisat2_sim/processing_index.csv").drop_duplicates("fire_id").set_index("fire_id")["country"]
scenes["country"] = np.where(scenes.source == "heo", "IT", scenes.scene.str.rsplit("_", n=1).str[0].map(country).fillna("?"))
print("PhiSat-2 scenes per country:", scenes[scenes.source == "phisat2_sim"].country.value_counts().to_dict())
print("scenes with an image but no v3 mask:", missing if missing else "none")
scenes.groupby("source").agg(scenes=("scene", "size"), px_m=("px_m", "median"), fresh_px=("fresh_px", "sum"),
                             old_px=("old_px", "sum"), first=("acq", "min"), last=("acq", "max"))
# expected: heo 35, phisat2_sim 383, s2_test 3

## 2 · HEO scenes we leave out

A scene stays unless something is technically wrong with it. Eleven of the 25 earlier HEO scenes have a problem we can see in their spectra: a sawtooth between the red-edge bands (RE2 far above RE1 and RE3, which no surface does; the IMH04 strips of 17 March also have snow and haze), striping, or bands out of range. The plot shows the median reflectance of clear land in every HEO scene: the kept ones look like vegetation and soil, a smooth rise from the red to the NIR.

The 7 L2A scenes (dashed) stay. They were picked for their clouds and for dark ground that is not burnt, such as volcanic soil, which is exactly what the model has to learn to leave alone. They are a different product level and their values run brighter than L1C; the model sees them as they are.

In [ ]:
# [3] scenes left out, and the clear-land spectrum of every HEO scene
EXCLUDE = {                                                # scene name starts with -> reason
    "IMH04_1CST__OPT8_20260317T123240": "sawtooth spectrum, snow and haze",
    "IMH04_1CST__OPT8_20260317T123336": "sawtooth spectrum, snow and haze",
    "IMH04_1CST__OPT8_20260317T123338": "sawtooth spectrum, snow and haze",
    "IMH04_1CST__OPT8_20260317T123341": "sawtooth spectrum, snow and haze",
    "IMH04_1CST__OPT8_20260317T123343": "sawtooth spectrum, snow and haze",
    "IMH09_1CST__OPT8_20260722T124549": "striping",
    "IMH04_1CST__OPT8_20260303T123520": "sawtooth spectrum",
    "IMH04_1CST__OPT8_20260317T123344": "sawtooth spectrum",
    "IMH02_1CST__OPT8_20260308T122749": "sawtooth spectrum",
    "IMH02_1CST__OPT8_20251115T104743": "sawtooth spectrum",
    "IMH08_1CST__OPT8_20260301T130916": "bands out of range",
}
USE_L2A = True        # L2A scenes stay: clouds and dark ground that is not burnt

def why_out(r):
    reason = next((why for k, why in EXCLUDE.items() if r["scene"].startswith(k)), "")
    if not reason and r["source"] == "heo" and r["level"] == "L2A" and not USE_L2A:
        reason = "L2A"
    return reason

scenes["excluded"] = scenes.apply(why_out, axis=1)

def clear_spectrum(r, decim=8):
    with rasterio.open(r.img) as s:
        bands = D.band_indexes(s.descriptions, s.count)
        h, w = max(s.height // decim, 1), max(s.width // decim, 1)
        a = s.read(bands, out_shape=(7, h, w), resampling=Resampling.nearest).astype(np.float32) / r.scale
    with rasterio.open(r.mask) as m:
        mk = m.read(1, out_shape=(h, w), resampling=Resampling.nearest)
    sel = mk == L.CLEAR
    return np.median(a[:, sel], axis=1) if sel.sum() > 100 else np.full(7, np.nan)

heo = scenes[scenes.source == "heo"].copy()
spec = np.stack([clear_spectrum(r) for r in heo.itertuples()])
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8), sharey=True)
for k, (title, sel) in enumerate([("kept", (heo.excluded == "") | (heo.excluded == "L2A")),
                                  ("left out for a problem in the data", ~heo.excluded.isin(["", "L2A"]))]):
    for sid, sp, lvl in zip(heo.scene[sel], spec[sel.values], heo.level[sel]):
        ax[k].plot(D.BAND_ORDER, sp, lw=1.3, alpha=0.85, ls="--" if lvl == "L2A" else "-",
                   label=f"{sid[:5]} {sid[17:32]}{' L2A' if lvl == 'L2A' else ''}")
    ax[k].set_title(f"HEO, {title} ({int(sel.sum())} scenes)", loc="left", fontsize=10)
    ax[k].legend(fontsize=6.5, ncol=2, loc="upper left")
ax[0].set_ylabel("median reflectance of clear land")
plt.tight_layout(); plt.show()
heo.assign(scene=heo.scene.str[:32])[["scene", "level", "excluded", "fresh_px", "old_px", "cloud_px", "geo_rms_m"]].sort_values("scene").round(1)

## 3 · Splits by ground

Scenes whose footprints overlap go in the same group, and so do HEO frames of the same pass (same spacecraft and orbit number in the name), even when they only touch. A whole group goes to one split, so different dates over the same ground, or two frames of one fire, are never split between training and test.

The model targets HEO, so it is validated and tested on HEO: two scenes for test and one for validation, the rest for training. Test: IMH09 2026-07-22 12:45:51 and IMH01 2026-08-15, two spacecraft over two areas, 21% of the HEO fresh burn; the >80% is measured on them. Validation: IMH03 2026-08-24, 2%. Training keeps 77%, including the IMH06 group (the most fresh burn and the best geolocation), IMH09 2026-08-01 (the most old burn) and the two IMH08 products, whose geolocation (12 and 14 m in their metadata, about 5 pixels) is fine for training but too poor for scoring. None of the three held-out scenes shares ground with a training scene. [5] warns about any test or validation scene whose metadata gives a geolocation error above 8 m; IMH09 has no metadata here, so [13] checks by eye that its labels sit on the burns. New HEO acquisitions can be added to the test set the same way, by name.

Every PhiSat-2 scene over the same ground as an HEO test or validation scene, or a Sentinel-2 test chip, is left out: it shows the same fires and fields, simulated from Sentinel-2, so it can't be in training. Only the scenes that touch that ground go, not their whole group. The other PhiSat-2 groups go at random to validation (10% of its fresh burn) and test (10%). The Sentinel-2 chips are only for testing.

In [ ]:
# [4] footprint groups, and the HEO groups with their fresh burn
use = scenes.excluded == ""
scenes["group"] = ""
for src, g in scenes[use].groupby("source"):
    ids = D.footprint_groups(g[["lon0", "lat0", "lon1", "lat1"]].values)
    if src == "heo":       # frames of the same pass (same spacecraft and orbit number) stay together even if they only touch
        ids = D.merge_groups(ids, (g.scene.str[:5] + "_" + g.scene.str.split("_").str[7]).values)
    scenes.loc[g.index, "group"] = [f"{src}_{i}" for i in ids]
print(scenes[use].groupby("source").agg(scenes=("scene", "size"), groups=("group", "nunique")).to_string())

hg = (scenes[use & (scenes.source == "heo")].groupby("group")
        .agg(scenes=("scene", lambda s: ", ".join(f"{x[:5]} {x[17:25]}" for x in s)), n=("scene", "size"),
             levels=("level", lambda s: ",".join(sorted(set(s)))), fresh_px=("fresh_px", "sum"), old_px=("old_px", "sum"),
             geo_rms_m=("geo_rms_m", "max"))
        .sort_values("fresh_px", ascending=False))
hg["fresh_%"] = (100 * hg.fresh_px / max(hg.fresh_px.sum(), 1)).round(1)
hg

In [ ]:
# [5] the split
# HEO held out (the start of the name is enough; a scene's whole group goes with it): two scenes for test, two
# spacecraft and two areas, 21% of the HEO fresh burn; one for validation, 2%. Training keeps 77%.
HEO_TEST = ["IMH09_1CST__OPT8_20260722T124551",   # 829 k fresh pixels, no geolocation metadata (see [13])
            "IMH01_1CST__OPT8_20260815T103337"]   # 194 k, geolocation 4.1 m
HEO_VAL = ["IMH03_1CST__OPT8_20260824T122543"]    # 108 k, geolocation 3.0 m
MAX_GEO_RMS = 8.0                   # metres: above this the labels can be off by several pixels, fine for training only

def group_of(prefix):
    g = scenes.loc[use & scenes.scene.str.startswith(prefix), "group"]
    if g.empty:
        raise ValueError(f"no scene in use starts with {prefix}")
    return g.iloc[0]

split_of = {group_of(p): "test" for p in HEO_TEST}
split_of.update({group_of(p): "val" for p in HEO_VAL})
for r in scenes[scenes.group.isin(split_of) & (scenes.geo_rms_m > MAX_GEO_RMS)].itertuples():
    print(f"warning: {r.scene[:32]} is held out but its geolocation error is {r.geo_rms_m:.1f} m")

is_heo, is_phi = scenes.source == "heo", scenes.source == "phisat2_sim"
scenes["split"] = "train"
scenes.loc[is_heo, "split"] = scenes.loc[is_heo, "group"].map(split_of).fillna("train")
scenes.loc[scenes.source == "s2_test", "split"] = "test"

# A PhiSat-2 scene over the same ground as an HEO test or validation scene, or a Sentinel-2 test chip, shows the same
# fires and fields simulated from Sentinel-2, so it can't be in training. Only the scenes that touch that ground are
# left out, not their whole group: around Sicily the groups chain over far more ground than the test scenes cover.
def overlapping(b, boxes, min_overlap=0.01):
    w = np.clip(np.minimum(b[2], boxes[:, 2]) - np.maximum(b[0], boxes[:, 0]), 0, None)
    h = np.clip(np.minimum(b[3], boxes[:, 3]) - np.maximum(b[1], boxes[:, 1]), 0, None)
    area = lambda x: (x[..., 2] - x[..., 0]) * (x[..., 3] - x[..., 1])
    return w * h > min_overlap * np.minimum(area(b), area(boxes))

held = scenes[(scenes.source != "phisat2_sim") & scenes.split.isin(["val", "test"])].reset_index(drop=True)
boxes = held[["lon0", "lat0", "lon1", "lat1"]].values
for i, r in scenes[is_phi & use].iterrows():
    hits = np.flatnonzero(overlapping(np.array([r["lon0"], r["lat0"], r["lon1"], r["lat1"]]), boxes)) if len(boxes) else []
    if len(hits):
        h = held.loc[hits[0]]
        scenes.loc[i, "excluded"] = f"same ground as the {SRC_NAME[h['source']]} {h['split']} scene {h['scene'][:32]}"
use = scenes.excluded == ""
off = is_phi & scenes.excluded.str.startswith("same ground")
print(f"PhiSat-2 scenes left out because they cover held-out ground: {int(off.sum())} "
      f"({scenes.loc[off, 'fresh_px'].sum():,} fresh pixels)")
display(scenes.loc[off].groupby("excluded").size().to_frame("PhiSat-2 scenes"))

# the other PhiSat-2 groups go at random to validation and test
w = scenes.loc[is_phi & use, "fresh_px"]
scenes.loc[is_phi & use, "split"] = D.random_group_split(scenes.loc[is_phi & use, "group"].values,
                                                         (w + 0.25 * w.mean()).values, val=0.1, test=0.1, seed=0)
scenes.loc[~use, "split"] = "excluded"
scenes.to_csv(f"{TABLES}/scenes.csv", index=False)

for s in ("test", "val"):
    names = sorted(scenes.scene[is_heo & (scenes.split == s)].str[:32].tolist())
    print(f"HEO {s}: {names if names else 'none, every HEO scene in use is in training'}")
tab = scenes.groupby(["source", "split"]).agg(scenes=("scene", "size"), fresh_px=("fresh_px", "sum"), old_px=("old_px", "sum"))
tab["fresh_%"] = (100 * tab.fresh_px / tab.groupby(level=0).fresh_px.transform("sum").clip(lower=1)).round(1)
tab

In [ ]:
# [6] where the splits are: all scenes, and HEO on its own
inuse_map = scenes.split != "excluded"
cx, cy = (scenes.lon0 + scenes.lon1) / 2, (scenes.lat0 + scenes.lat1) / 2
far = inuse_map & (((cx - cx[inuse_map].median()).abs() > 25) | ((cy - cy[inuse_map].median()).abs() > 15))
for r, x_, y_ in zip(scenes[far].itertuples(), cx[far], cy[far]):
    print(f"not drawn, far from the rest: {r.scene[:32]} ({r.source}, {r.split}) at {x_:.1f} E, {y_:.1f} N")
inuse_map = inuse_map & ~far
fig, ax = plt.subplots(1, 2, figsize=(17, 6.5))
for k, sel in enumerate([inuse_map & (scenes.source != "s2_test"), inuse_map & (scenes.source == "heo")]):
    for r in scenes[sel].itertuples():
        filled = r.source == "heo"
        ax[k].add_patch(Rectangle((r.lon0, r.lat0), r.lon1 - r.lon0, r.lat1 - r.lat0, lw=1.2 if filled else 0.6,
                                  edgecolor=SPLIT_COLOR[r.split], facecolor=SPLIT_COLOR[r.split] if filled else "none",
                                  alpha=0.55 if filled else 0.9))
    sub = scenes[sel]
    pad = 0.3 if k == 0 else 0.1
    ax[k].set_xlim(sub.lon0.min() - pad, sub.lon1.max() + pad); ax[k].set_ylim(sub.lat0.min() - pad, sub.lat1.max() + pad)
    ax[k].set_aspect(1 / np.cos(np.radians(sub[["lat0", "lat1"]].values.mean())))
    ax[k].set_xlabel("longitude"); ax[k].set_ylabel("latitude")
ax[0].set_title("all scenes: HEO filled, PhiSat-2 outlined", loc="left", fontsize=10)
ax[1].set_title("HEO", loc="left", fontsize=10)
for r in scenes[inuse_map & (scenes.source == "heo")].itertuples():
    ax[1].annotate(f"{r.scene[:5]} {r.scene[21:25]}", ((r.lon0 + r.lon1) / 2, (r.lat0 + r.lat1) / 2), fontsize=6.5,
                   ha="center", color=INK2)
fig.legend(handles=[Patch(facecolor=SPLIT_COLOR[s], edgecolor=SPLIT_COLOR[s], label=s) for s in ("train", "val", "test")],
           loc="lower center", ncol=3)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

## 4 · Are the sources on the same scale?

The model only works on HEO if a burn in HEO looks like a burn in the simulated PhiSat-2 scenes. Below: reflectance per band and source, for clear land (the middle 98% and the median) and for every pixel with data (the top 0.1%, where clouds and saturated pixels are). Reflectance above 1 is not physical outside bright clouds.

In [ ]:
# [7] reflectance per band and source (HEO: every scene in use, PhiSat-2: 60 scenes, Sentinel-2: the 3 chips)
def sample_refl(r, decim=8, n=20000, seed=0):
    with rasterio.open(r["img"]) as s:
        bands = D.band_indexes(s.descriptions, s.count)
        h, w = max(s.height // decim, 1), max(s.width // decim, 1)
        a = s.read(bands, out_shape=(7, h, w), resampling=Resampling.nearest).astype(np.float32) / r["scale"]
    with rasterio.open(r["mask"]) as m:
        mk = m.read(1, out_shape=(h, w), resampling=Resampling.nearest)
    rng, out = np.random.default_rng(seed), {}
    for name, sel in (("clear land", mk == L.CLEAR), ("all with data", mk != L.NODATA)):
        idx = np.flatnonzero(sel)
        if len(idx) > n:
            idx = rng.choice(idx, n, replace=False)
        out[name] = a.reshape(7, -1)[:, idx]
    return out

inuse = scenes[scenes.split != "excluded"]
phi_inuse = inuse[inuse.source == "phisat2_sim"]
picked = pd.concat([inuse[inuse.source == "heo"], phi_inuse.sample(min(60, len(phi_inuse)), random_state=0),
                    inuse[inuse.source == "s2_test"]])
samples = {}
for _, r in picked.iterrows():
    for k, v in sample_refl(r, decim=1 if r["source"] == "s2_test" else 8).items():
        samples.setdefault((r["source"], k), []).append(v)
samples = {k: np.concatenate(v, axis=1) for k, v in samples.items()}

q = pd.DataFrame([dict(source=src, pixels=k, band=b, p1=np.percentile(v[i], 1), p50=np.median(v[i]),
                       p99=np.percentile(v[i], 99), p99_9=np.percentile(v[i], 99.9), max=v[i].max(),
                       above_1=100 * (v[i] > 1).mean())
                  for (src, k), v in samples.items() for i, b in enumerate(D.BAND_ORDER)])

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
x = np.arange(7)
srcs = [s for s in ("heo", "phisat2_sim", "s2_test") if (s, "clear land") in samples]
for i, src in enumerate(srcs):
    g = q[(q.source == src) & (q.pixels == "clear land")].set_index("band").loc[D.BAND_ORDER]
    off = (i - (len(srcs) - 1) / 2) * 0.22
    ax[0].errorbar(x + off, g.p50, yerr=[g.p50 - g.p1, g.p99 - g.p50], fmt="o", ms=5, capsize=3,
                   color=SRC_COLOR[src], label=SRC_NAME[src])
    g = q[(q.source == src) & (q.pixels == "all with data")].set_index("band").loc[D.BAND_ORDER]
    ax[1].plot(x + off, g.p99_9, "o", color=SRC_COLOR[src], label=f"{SRC_NAME[src]} 99.9%")
    ax[1].plot(x + off, g["max"], "x", color=SRC_COLOR[src], label=f"{SRC_NAME[src]} max")
for a in ax:
    a.set_xticks(x); a.set_xticklabels(D.BAND_ORDER)
ax[0].set_ylabel("reflectance"); ax[0].set_title("clear land: median, 1% to 99%", loc="left", fontsize=10)
ax[1].axhline(1, color=INK2, lw=0.8, ls=":")
ax[1].set_title("every pixel with data: top 0.1% and maximum", loc="left", fontsize=10)
ax[0].legend(loc="upper left"); ax[1].legend(loc="upper left", fontsize=7, ncol=3)
plt.tight_layout(); plt.show()
q.pivot_table(index=["pixels", "band"], columns="source", values=["p50", "p99_9", "above_1"]).round(3)

In [ ]:
# [8] normalisation: clip reflectance at CLIP, then (x - mean) / std per band. Mean and std come from the
# training scenes (every pixel with data), with HEO and PhiSat-2 counted equally. Saved to normalisation.json.
CLIP = 1.2        # TOA reflectance above this is bright cloud or saturation; it only has to stay bright

tr = scenes[scenes.split == "train"]
stats = {}
for src in ("heo", "phisat2_sim"):
    rows_ = tr[tr.source == src]
    rows_ = rows_.sample(min(len(rows_), 60), random_state=1)
    v = np.clip(np.concatenate([sample_refl(r, decim=8)["all with data"] for _, r in rows_.iterrows()], axis=1), 0, CLIP)
    stats[src] = (v.mean(1), v.std(1), (v >= CLIP).mean() * 100)
means = np.stack([m for m, _, _ in stats.values()])
mean = means.mean(0)
std = np.sqrt(np.mean([s ** 2 for _, s, _ in stats.values()], axis=0) + means.var(0))    # std of the 50/50 mixture
norm = {"bands": D.BAND_ORDER, "clip": CLIP, "mean": mean.round(5).tolist(), "std": std.round(5).tolist(),
        "scale": {"heo": 4094.0, "phisat2_sim": 10000.0, "s2_test": 10000.0}}
with open(f"{TABLES}/normalisation.json", "w") as f:
    json.dump(norm, f, indent=2)
for src, (_, _, clipped) in stats.items():
    print(f"{SRC_NAME[src]}: {clipped:.3f}% of the values reach the clip at {CLIP}")
pd.DataFrame({**{f"mean {SRC_NAME[s]}": m for s, (m, _, _) in stats.items()}, "mean": mean, "std": std},
             index=D.BAND_ORDER).T.round(4)

## 5 · Training windows

Every training window is 256 x 256 pixels. Half of them come from HEO, which is what the model sees in orbit, half from the simulated PhiSat-2 scenes, which have most of the burns. The ground resolution is drawn on a log scale: 1 to 10 m for HEO (finer than its 2.5 m is resampled up, so the model also gets used to 1 m data), 2.5 to 10 m for PhiSat-2. For HEO, 40% of the windows are centred on fresh burn and 15% on old burn (moved off-centre a bit, so a burn is not always in the middle), and the other 45% anywhere with data: every HEO scene gives those, so all its clear land is used. For PhiSat-2 it is 45% fresh, 20% old and 35% anywhere, and those windows centred anywhere, which are mostly clear land, come only from Italian scenes, the ground HEO covers. Burn-centred PhiSat-2 windows still come from every country. Augmentation, all on the image only: rotations and flips, a gain for all bands (0.9 to 1.1) and per band (0.95 to 1.05) for calibration between spacecraft, bands moved by up to 1 pixel half of the time because L1B bands are not co-registered, some blur and some noise.

In [ ]:
# [9] where training windows can be centred: fresh burn, old burn, anywhere with data (mask at 1/8 resolution)
DECIM = 8
tr = scenes[scenes.split == "train"]
points = {}
for k, r in enumerate(tr.itertuples(), 1):
    with rasterio.open(r.mask) as m:
        mk = m.read(1, out_shape=(max(m.height // DECIM, 1), max(m.width // DECIM, 1)), resampling=Resampling.nearest)
    points[r.scene] = D.sample_points(mk, DECIM, max_points=4000, rng=np.random.default_rng(k))
    if k % 50 == 0 or k == len(tr):
        print(f"  {k}/{len(tr)} training scenes")
D.save_points(points, f"{TABLES}/window_points.npz")
npts = pd.DataFrame([{"scene": s, **{k: len(v) for k, v in d.items()}} for s, d in points.items()]).merge(scenes[["scene", "source"]])
npts.groupby("source").agg(scenes=("scene", "size"), with_fresh=("fresh", lambda x: int((x > 0).sum())),
                           with_old=("old", lambda x: int((x > 0).sum())))

In [ ]:
# [10] the settings, and 16 training windows to look at (top: NIR, red, green; bottom: target, pink = left out)
SIZE = 256
SOURCE_WEIGHTS = {"heo": 0.5, "phisat2_sim": 0.5}
GSD_RANGE = {"heo": (1.0, 10.0), "phisat2_sim": (2.5, 10.0)}
P_FRESH = {"heo": 0.40, "phisat2_sim": 0.45}       # chance a window is centred on fresh burn
P_OLD = {"heo": 0.15, "phisat2_sim": 0.20}         # on old burn; the rest are centred anywhere with data
VALID_COUNTRIES = ["IT"]                           # PhiSat-2 windows centred anywhere (mostly clear) only from these

train_scenes = scenes[scenes.split == "train"]
valid_scenes = {"phisat2_sim": sorted(train_scenes.scene[(train_scenes.source == "phisat2_sim")
                                                         & train_scenes.country.isin(VALID_COUNTRIES)])}
print(f"PhiSat-2 training scenes that give windows anywhere: {len(valid_scenes['phisat2_sim'])} of "
      f"{int((train_scenes.source == 'phisat2_sim').sum())} ({', '.join(VALID_COUNTRIES)})")
ds = D.TrainWindows(train_scenes, points, norm, size=SIZE, length=4000, source_weights=SOURCE_WEIGHTS,
                    gsd_range=GSD_RANGE, p_fresh=P_FRESH, p_old=P_OLD, valid_scenes=valid_scenes, augment=True, seed=0)

def false_colour(refl):
    a = refl[[6, 2, 1]]
    valid = (a > 0).all(0)
    out = np.ones(a.shape[1:] + (3,), np.float32)
    for i in range(3):
        lo, hi = np.percentile(a[i][valid], (2, 98)) if valid.sum() > 10 else (0, 1)
        out[..., i] = np.clip((a[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    out[~valid] = 1
    return out

def show_windows(items, ncol=8):
    nrow = 2 * int(np.ceil(len(items) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.45 * nrow))
    for a in axes.ravel():
        a.axis("off")
    for k, d in enumerate(items):
        r, c = 2 * (k // ncol), k % ncol
        axes[r, c].imshow(false_colour(d["refl"]))
        axes[r, c].set_title(f"{SRC_NAME[d['source']]} {d['gsd']:.1f} m\n{d.get('kind', '')}", fontsize=8)
        axes[r + 1, c].imshow(np.ma.masked_less(d["y"], 0), cmap=TRAIN_CMAP, vmin=-0.5, vmax=5.5, interpolation="nearest")
    fig.legend(handles=[Patch(facecolor=TRAIN_CMAP(i), edgecolor="#8a8984", label=n) for i, n in enumerate(L.TRAIN_NAMES)]
               + [Patch(facecolor=L.V3_COLORS[L.IGNORE], edgecolor="#8a8984", label="left out")], loc="lower center", ncol=7)
    plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

show_windows([ds.get(i, with_info=True) for i in range(16)])

In [ ]:
# [11] what the model will see: class shares in 600 training windows, and loss weights from them
# (no augmentation; a few minutes. The time per window says whether reading will keep up with the GPU)
import time
ds_plain = D.TrainWindows(train_scenes, points, norm, size=SIZE, length=600, source_weights=SOURCE_WEIGHTS,
                          gsd_range=GSD_RANGE, p_fresh=P_FRESH, p_old=P_OLD, valid_scenes=valid_scenes,
                          augment=False, seed=1)
cnt = {s: np.zeros(7, np.int64) for s in SOURCE_WEIGHTS}
t0 = time.time()
for i in range(len(ds_plain)):
    d = ds_plain.get(i, with_info=True)
    cnt[d["source"]] += np.bincount(d["y"].ravel() + 1, minlength=7)
    if (i + 1) % 200 == 0:
        print(f"  {i + 1}/{len(ds_plain)} windows")
freq = pd.DataFrame({SRC_NAME[s]: c[1:] for s, c in cnt.items()}, index=L.TRAIN_NAMES)
freq["all"] = freq.sum(axis=1)
share = 100 * freq / freq.sum()
w = (share["all"] / 100).clip(lower=1e-4) ** -0.5                  # 1 / sqrt(frequency), mean 1
CLASS_WEIGHTS = (w / w.mean()).round(3)
display(pd.concat({"% of labelled pixels": share.round(2), "loss weight": CLASS_WEIGHTS.to_frame("all")}, axis=1))
print("left out (ignore and no data): " + ", ".join(f"{SRC_NAME[s]} {100 * c[0] / c.sum():.1f}%" for s, c in cnt.items()))
print(f"reading: {1000 * (time.time() - t0) / len(ds_plain):.0f} ms per window in one process")

fig, ax = plt.subplots(figsize=(10, 3.8))
xs, cols = np.arange(len(L.TRAIN_NAMES)), [c for c in share.columns if c != "all"]
for i, c in enumerate(cols):
    src = {v: k for k, v in SRC_NAME.items()}[c]
    bars = ax.bar(xs + (i - (len(cols) - 1) / 2) * 0.4, share[c].clip(lower=1e-3), 0.37, color=SRC_COLOR[src], label=c)
    ax.bar_label(bars, labels=[f"{v:.2g}" for v in share[c]], fontsize=7, color=INK2, padding=2)
ax.set_yscale("log"); ax.set_xticks(xs); ax.set_xticklabels(L.TRAIN_NAMES)
ax.set_ylabel("% of labelled pixels (log scale)"); ax.legend(loc="upper right")
ax.set_title("class shares in training windows", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 6 · Validation and test windows

Validation and test scenes are cut into windows that cover the whole scene, at a few resolutions: HEO at 2.5, 5 and 10 m, PhiSat-2 at 4.75 and 10 m, the Sentinel-2 chips at 10 m. Windows with less than 30% data are dropped. Scores are computed on these windows, always the same ones. Cell [13] checks by eye, on every HEO scene with fresh burn, that the labels sit on the burns.

In [ ]:
# [12] the windows, saved to windows_val.csv and windows_test.csv
EVAL_GSD = {"heo": [2.5, 5.0, 10.0], "phisat2_sim": [4.75, 10.0], "s2_test": [10.0]}
rows = []
for r in scenes[scenes.split.isin(["val", "test"])].itertuples():
    dec = 1 if r.source == "s2_test" else 4
    with rasterio.open(r.mask) as m:
        mk = m.read(1, out_shape=(max(m.height // dec, 1), max(m.width // dec, 1)), resampling=Resampling.nearest)
    for gsd in EVAL_GSD[r.source]:
        for r0, c0, n in D.grid_boxes(r.height, r.width, r.px_m, gsd, SIZE):
            sub = mk[r0 // dec:(r0 + n) // dec, c0 // dec:(c0 + n) // dec]
            valid = float((sub != L.NODATA).mean()) if sub.size else 0.0
            if valid < 0.3:
                continue
            rows.append(dict(scene=r.scene, source=r.source, split=r.split, gsd=round(n * r.px_m / SIZE, 2),
                             r0=r0, c0=c0, n=n, valid=round(valid, 3), fresh=round(float((sub == L.FRESH).mean()), 4),
                             old=round(float((sub == L.OLD).mean()), 4)))
windows = pd.DataFrame(rows)
for s in ("val", "test"):
    windows[windows.split == s].to_csv(f"{TABLES}/windows_{s}.csv", index=False)
windows.groupby(["split", "source", "gsd"]).agg(windows=("scene", "size"), scenes=("scene", "nunique"),
                                                 with_fresh=("fresh", lambda x: int((x > 0).sum())))

In [ ]:
# [13] do the labels sit on the burns? For every HEO scene with fresh burn, the 512 x 512 window at full resolution
# (2.5 m) with the most fresh burn: NIR, red, green with the label outlines on top (yellow fresh, cyan old).
# The outline should follow the dark scar. A shift of a few pixels in one direction is a geolocation error:
# look first at the test and validation scenes, then at IMH08 0815 and 0906 (12 and 14 m in their metadata).
from scipy import ndimage
N, DEC = 512, 8
rows = []
for r in scenes[(scenes.source == "heo") & (scenes.split != "excluded") & (scenes.fresh_px > 0)].itertuples():
    with rasterio.open(r.mask) as m:
        mk = m.read(1, out_shape=(m.height // DEC, m.width // DEC), resampling=Resampling.nearest)
    f = ndimage.uniform_filter((mk == L.FRESH).astype(np.float32), size=N // DEC, mode="constant")
    i, j = np.unravel_index(np.argmax(f), f.shape)
    rows.append(dict(scene=r.scene, split=r.split, r0=int(np.clip(i * DEC - N // 2, 0, r.height - N)),
                     c0=int(np.clip(j * DEC - N // 2, 0, r.width - N)), n=N, geo=r.geo_rms_m))
best = pd.DataFrame(rows)
best = best.assign(o=best.split.map({"test": 0, "val": 1}).fillna(2)).sort_values(["o", "scene"]).drop(columns="o")
fw = D.FixedWindows(best, scenes, norm, size=N)
ncol = 4
nrow = int(np.ceil(len(fw) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.6 * nrow), squeeze=False)
for a in axes.ravel():
    a.axis("off")
for i in range(len(fw)):
    x, y, refl, _ = fw.get(i, with_refl=True)
    info, a = fw.info(i), axes.ravel()[i]
    a.imshow(false_colour(refl))
    for cls, colour in ((1, "#ffd400"), (2, "#00e5ff")):
        if (y == cls).any():
            a.contour((y == cls).astype(float), levels=[0.5], colors=[colour], linewidths=1.0)
    geo = "no metadata" if np.isnan(info.geo) else f"geolocation {info.geo:.1f} m"
    a.set_title(f"{info.split}: {info.scene[:5]} {info.scene[17:32]}\n{geo}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# [14] everything decided here, in one file the training notebooks read
config = dict(size=SIZE, source_weights=SOURCE_WEIGHTS, gsd_range=GSD_RANGE, p_fresh=P_FRESH, p_old=P_OLD,
              valid_countries=VALID_COUNTRIES, valid_scenes=valid_scenes,
              augment=D.AUG_DEFAULTS, class_names=L.TRAIN_NAMES, class_weights=CLASS_WEIGHTS.tolist(),
              eval_gsd=EVAL_GSD, excluded=EXCLUDE, use_l2a=USE_L2A,
              heo_test=sorted(scenes.scene[is_heo & (scenes.split == "test")].tolist()),
              heo_val=sorted(scenes.scene[is_heo & (scenes.split == "val")].tolist()),
              files=dict(scenes=f"{TABLES}/scenes.csv", normalisation=f"{TABLES}/normalisation.json",
                         points=f"{TABLES}/window_points.npz", windows_val=f"{TABLES}/windows_val.csv",
                         windows_test=f"{TABLES}/windows_test.csv"),
              created=pd.Timestamp.now().isoformat())
with open(f"{TABLES}/dataset_config.json", "w") as f:
    json.dump(config, f, indent=2, default=str)
for p in config["files"].values():
    print(f"{p:<40} {os.path.getsize(p) / 1e6:6.2f} MB")

## How the training notebooks use this

```python
import burnscar.dataset as D
cfg = json.load(open("data/tables/dataset_config.json"))
scenes = pd.read_csv(cfg["files"]["scenes"])
norm = json.load(open(cfg["files"]["normalisation"]))
train = D.TrainWindows(scenes[scenes.split == "train"], D.load_points(cfg["files"]["points"]), norm,
                       size=cfg["size"], length=8000, source_weights=cfg["source_weights"], gsd_range=cfg["gsd_range"],
                       p_fresh=cfg["p_fresh"], p_old=cfg["p_old"], valid_scenes=cfg["valid_scenes"])
val = D.FixedWindows(pd.read_csv(cfg["files"]["windows_val"]), scenes, norm, size=cfg["size"])
```

Both work with PyTorch's `DataLoader` as they are. Every DataLoader worker opens its own files.